# ComfyUI + Z-Image Turbo - 2560x1440 生成

Google Colabで高解像度画像を生成します

## 手順
1. ランタイム → GPU T4以上を選択
2. すべてのセルを実行
3. 生成完了後、画像をダウンロード

In [ ]:
# GPU確認
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram:.1f} GB")

In [ ]:
# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt -q

In [ ]:
# モデルダウンロード（10GB）
import os
from huggingface_hub import hf_hub_download

models_dir = "/content/ComfyUI/models/checkpoints"
os.makedirs(models_dir, exist_ok=True)

print("Downloading Z-Image Turbo (10GB)...")
hf_hub_download(
    repo_id="SeeSee21/Z-Image-Turbo-FP8-AIO",
    filename="z-image-turbo-fp8-aio.safetensors",
    local_dir=models_dir,
    local_dir_use_symlinks=False
)
print("Done!")

In [ ]:
# configファイル
import subprocess
os.makedirs("/content/ComfyUI/models/configs", exist_ok=True)
subprocess.run(["wget", "-q", "-O", 
    "/content/ComfyUI/models/configs/v1-inference.yaml",
    "https://raw.githubusercontent.com/comfyanonymous/ComfyUI/master/configs/v1-inference.yaml"
], check=True)
print("Config OK")

In [ ]:
# 画像生成（2560x1440）
import requests
import time
import json

# ワークフロー
workflow = {
    "1": {"inputs": {"ckpt_name": "z-image-turbo-fp8-aio.safetensors", "config_name": "v1-inference.yaml"}, "class_type": "CheckpointLoader"},
    "2": {"inputs": {"text": "Professional stock photo of Japanese businessman in suit walking through Tokyo Shibuya crossing at night, neon lights, crowd, cinematic lighting, 4K quality", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "3": {"inputs": {"text": "", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "4": {"inputs": {"width": 2560, "height": 1440, "batch_size": 1}, "class_type": "EmptyLatentImage"},
    "5": {"inputs": {"model": ["1", 0], "seed": 42, "steps": 8, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "positive": ["2", 0], "negative": ["3", 0], "latent_image": ["4", 0], "denoise": 1.0}, "class_type": "KSampler"},
    "6": {"inputs": {"samples": ["5", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
    "7": {"inputs": {"images": ["6", 0], "filename_prefix": "result_2560x1440"}, "class_type": "SaveImage"}
}

print("Generating 2560x1440 image...")
result = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": workflow}).json()
prompt_id = result['prompt_id']

# 完了待ち
for i in range(300):
    time.sleep(1)
    hist = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}").json()
    if prompt_id in hist and hist[prompt_id]['status']['completed']:
        print("Complete!")
        break
    if i % 30 == 0:
        print(f"Progress: {i}s...")

print("Done!")

In [ ]:
# 画像ダウンロード
from google.colab import files
import glob

output_files = glob.glob("/content/ComfyUI/output/result_2560x1440_*.png")
if output_files:
    latest = max(output_files, key=os.path.getmtime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No output found")